In [ ]:
reset()
from SageNP import NewmanPenrose

#########################################################################################################
# Define 4-dim. the manifold:
MyManifold = Manifold(4 , 'MyManifold', r'\mathcal{Man}')
MyCoordinates.<t,r,th,ph> = MyManifold.chart(r't r th:\vartheta ph:\varphi')


def noargs(name):
    return lambda *args: name


U = function("U",print_latex_func=noargs('U'),imag_part_func=0)(r, th, ph)
R = function("R",print_latex_func=noargs('R'),imag_part_func=0)(t)
X = function("X",print_latex_func=noargs('X'),imag_part_func=0)(t,r,th,ph)

X=(1+U/R)^-2
metric_functions=[("R(t)",R), ("X",(1+U/R)^-2), ("U(r,th,ph)",U)]

Sigma = r

lvec=[X**(1/2)*sqrt(1/2), X**(-1/2)*sqrt(1/2)*R,  0, 0]
nvec=[X**(1/2)*sqrt(1/2), -X**(-1/2)*sqrt(1/2)*R,  0, 0]
mvec=[0, 0, Sigma*X**(-1/2)*R*sqrt(1/2), Sigma*X**(-1/2)*R*sqrt(1/2)*I*sin(th)]
mbarvec=[0, 0, Sigma*X**(-1/2)*R*sqrt(1/2), Sigma*X**(-1/2)*R*sqrt(1/2)*-I*sin(th)]

f=MyManifold.scalar_field(function('f')(*MyCoordinates))

met=NewmanPenrose(MyManifold, MyCoordinates, lvec, nvec, mvec, mbarvec,'covariant')

#met.calculate_allNP()
met.calculate_spincoefficients()
met.calculate_Weyl()
met.calculate_Ricci()
met.calculate_NPeq()
met.calculate_Bianchi()
met.Petrov_fromWeyl()

#met.show_allNP()
met.show_spincoefficients()
met.show_Weyl()
met.show_Ricci()
met.show_NPeq()
met.show_Bianchi()



'The metric:'

g = -R(t)^2/(R(t)^2 + 2*R(t)*U(r, th, ph) + U(r, th, ph)^2) dt⊗dt + (R(t)^2 + 2*R(t)*U(r, th, ph) + U(r, th, ph)^2) dr⊗dr + (r^2*R(t)^2 + 2*r^2*R(t)*U(r, th, ph) + r^2*U(r, th, ph)^2) dth⊗dth + (r^2*R(t)^2 + 2*r^2*R(t)*U(r, th, ph) + r^2*U(r, th, ph)^2)*sin(th)^2 dph⊗dph

Calculating spin coefficients...
Calculating Weyl components...
Calculating Ricci components...
Calculating NP equations...
Calculating Bianchi identities...


KeyboardInterrupt: ECL says: Console interrupt.

In [ ]:
from pathlib import Path
fout = open('21.30_k0.tex', "w")

def fitEq(expr):
    try:
        expr = expr.expand()
    except Exception:
        pass
    return latex(expr)
def safe_expr(x):
    return x.expr() if hasattr(x, "expr") else x

def is_zero_expr(expr):
    expr = safe_expr(expr)
    try:
        return bool(expr.simplify_full() == 0)
    except Exception:
        try:
            return str(expr) == "0"
        except Exception:
            return False

def clean_metric_latex(metric_display):
    metric_ltx = fitEq(metric_display)
    if metric_ltx.startswith("g = "):
        metric_ltx = metric_ltx[4:]
    elif metric_ltx.startswith("= "):
        metric_ltx = metric_ltx[2:]
    elif metric_ltx.startswith("="):
        metric_ltx = metric_ltx[1:]
    return metric_ltx.strip()

def directional_operator_latex(vector_field, coords):
    terms = []
    for i, coord in enumerate(coords):
        coeff = safe_expr(vector_field[i])
        if is_zero_expr(coeff):
            continue
        coeff_ltx = fitEq(coeff)
        coord_ltx = latex(coord)

        if coeff_ltx == "1":
            terms.append(r"\partial_{" + coord_ltx + "}")
        elif coeff_ltx == "-1":
            terms.append(r"-\partial_{" + coord_ltx + "}")
        else:
            terms.append(coeff_ltx + r"\,\partial_{" + coord_ltx + "}")

    if not terms:
        return "0"

    return " + ".join(terms).replace("+ -", "- ")

def latex_write(fout, lhs, rhs_expr):
    """
    Uzun LaTeX ifadelerini \\frac sinirlarinda keserek yazar.
    Kisa ifadeler (<5500 karakter) dogrudan yazilir.
    Uzun ifadeler uygun bir \\frac noktasindan kesilip {} \\dots ile bitirilir.
    """
    my_expression = str(rhs_expr)

    if len(my_expression) < 5500:
        fout.write(r"\begin{dmath*}" + "\n")
        fout.write(lhs + " = " + my_expression + "\n")
        fout.write(r"\end{dmath*}" + "\n\n")
    else:
        count_oparan = 0
        count_cparan = 0
        last_frac_locations = []
        for i in range(min(5000, len(my_expression))):
            if my_expression[i] == "{":
                count_oparan += 1
            elif my_expression[i] == "}":
                count_cparan += 1
            if my_expression[i:i+5] == r"\frac" and count_oparan == count_cparan:
                last_frac_locations.append(i)

        if len(last_frac_locations) > 2:
            fout.write(r"\begin{dmath*}" + "\n")
            fout.write(lhs + " = " + my_expression[:last_frac_locations[-1]] + r"{} \dots" + "\n")
            fout.write(r"\end{dmath*}" + "\n\n")
        else:
            i = 5000
            while i < min(200000, len(my_expression)):
                if my_expression[i] == "{":
                    count_oparan += 1
                elif my_expression[i] == "}":
                    count_cparan += 1
                if len(my_expression) <= i + 5:
                    # Ifade tamamen taranabildi, hepsini yaz
                    fout.write(r"\begin{dmath*}" + "\n")
                    fout.write(lhs + " = " + my_expression + "\n")
                    fout.write(r"\end{dmath*}" + "\n\n")
                    return
                if my_expression[i:i+5] == r"\frac" and count_oparan == count_cparan:
                    last_frac_locations.append(i)
                    if len(last_frac_locations) > 2:
                        break
                i += 1

            if not last_frac_locations or last_frac_locations[-1] == 0:
                # Uygun kesme noktasi bulunamadi, yine de yaz
                fout.write(r"\begin{dmath*}" + "\n")
                fout.write(lhs + " = " + my_expression + "\n")
                fout.write(r"\end{dmath*}" + "\n\n")
            else:
                fout.write(r"\begin{dmath*}" + "\n")
                fout.write(lhs + " = " + my_expression[:last_frac_locations[-1]] + r"{} \dots" + "\n")
                fout.write(r"\end{dmath*}" + "\n\n")


def write_dmath(fout, lhs, rhs):
    """latex_write ile uzun ifade truncation destekli yazim."""
    latex_write(fout, lhs, rhs)

def write_display_line(fout, text):
    fout.write("\\begin{dmath*}\n")
    fout.write(text + "\n")
    fout.write("\\end{dmath*}\n\n")

def write_section(fout, title):
    fout.write(r"\vspace{0.45cm}" + "\n")
    fout.write(r"{\Large\bfseries " + title + "}\n\n")

def nonzero_tensor_components(components, coords):
    pieces = []
    for comp, coord in zip(components, coords):
        expr = safe_expr(comp)
        if is_zero_expr(expr):
            continue
        pieces.append(r"\left(" + fitEq(expr) + r"\right)\,d" + latex(coord))
    if not pieces:
        return "0"
    return " + ".join(pieces)

def nonzero_vector_components(components, coords):
    pieces = []
    for comp, coord in zip(components, coords):
        expr = safe_expr(comp)
        if is_zero_expr(expr):
            continue
        pieces.append(r"\left(" + fitEq(expr) + r"\right)\,d" + latex(coord))
    if not pieces:
        return "0"
    return " + ".join(pieces)

def write_named_exprs(fout, title, items, skip_zero=False):
    write_section(fout, title)
    wrote_any = False
    for lhs, expr in items:
        if skip_zero and is_zero_expr(expr):
            continue
        write_dmath(fout, lhs, fitEq(safe_expr(expr)))
        wrote_any = True
    if skip_zero and not wrote_any:
        fout.write(r"\textit{All expressions in this section vanish.}" + "\n\n")

def write_equation_section_with_summary(fout, title, items):
    nonzero_items = []
    vanished = False

    for lhs, expr in items:
        if is_zero_expr(expr):
            vanished = True
        else:
            nonzero_items.append((lhs, expr))

    write_section(fout, title)

    for lhs, expr in nonzero_items:
        write_dmath(fout, lhs, fitEq(safe_expr(expr)))

    if vanished:
        fout.write(r"\textit{Equations are vanished.}" + "\n\n")
    else:
        fout.write(r"\textit{Equations are not vanished.}" + "\n\n")


metric_name = Path(fout.name).stem
metric_title = f"Stephani et al., Equation ({metric_name})"



tmpstr = r'''\documentclass[landscape]{article}
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage[a4paper,landscape,margin=1.6cm]{geometry}
\usepackage[fleqn]{amsmath}
\usepackage{amssymb}
\usepackage{mathtools}
\usepackage{breqn}
\usepackage{xcolor}
\usepackage{lmodern}
\usepackage{microtype}

\setlength{\mathindent}{0pt}
\setlength{\parindent}{0pt}
\setlength{\parskip}{5pt}

\begin{document}
\raggedright

{\Huge\bfseries Newman-Penrose Formalism Calculations}

\vspace{0.2cm}
{\LARGE ''' + metric_title + r'''}

'''
fout.write(tmpstr)

write_section(fout, "Metric")
write_dmath(fout, r"ds^2", clean_metric_latex(met.g.display()))

write_section(fout, "Coordinates")
write_dmath(fout, r"\left(x^\mu\right)", fitEq(MyCoordinates[:]))

# Metrik fonksiyonları: eğer tanımlanmışsa yaz
try:
    if metric_functions:
        write_section(fout, "Metric Functions")
        for lhs, expr in metric_functions:
            write_dmath(fout, lhs, fitEq(expr))
except NameError:
    pass


write_section(fout, "Null Tetrad")

fout.write(r"{\large\itshape Covariant components}" + "\n\n")
write_dmath(fout, "l", nonzero_tensor_components([met.lNP[i] for i in range(4)], MyCoordinates))
write_dmath(fout, "n", nonzero_tensor_components([met.nNP[i] for i in range(4)], MyCoordinates))
write_dmath(fout, "m", nonzero_tensor_components([met.mNP[i] for i in range(4)], MyCoordinates))
write_dmath(fout, r"\overline{m}", nonzero_tensor_components([met.mbarNP[i] for i in range(4)], MyCoordinates))

fout.write(r"{\large\itshape Contravariant components}" + "\n\n")
write_dmath(fout, "l", nonzero_vector_components([met.lupNP[i] for i in range(4)], MyCoordinates))
write_dmath(fout, "n", nonzero_vector_components([met.nupNP[i] for i in range(4)], MyCoordinates))
write_dmath(fout, "m", nonzero_vector_components([met.mupNP[i] for i in range(4)], MyCoordinates))
write_dmath(fout, r"\overline{m}", nonzero_vector_components([met.mbarupNP[i] for i in range(4)], MyCoordinates))

write_section(fout, "Directional Derivatives")
write_dmath(fout, "D", directional_operator_latex(met.lupNP, MyCoordinates[:]))
write_dmath(fout, r"\Delta", directional_operator_latex(met.nupNP, MyCoordinates[:]))
write_dmath(fout, r"\delta", directional_operator_latex(met.mupNP, MyCoordinates[:]))
write_dmath(fout, r"\overline{\delta}", directional_operator_latex(met.mbarupNP, MyCoordinates[:]))

write_named_exprs(fout, "Spin Coefficients", [
    (r"\rho", met.rhoNP),
    (r"\sigma", met.sigmaNP),
    (r"\kappa", met.kappaNP),
    (r"\mu", met.muNP),
    (r"\lambda", met.lambdaNP),
    (r"\nu", met.nuNP),
    (r"\alpha", met.alphaNP),
    (r"\beta", met.betaNP),
    (r"\tau", met.tauNP),
    (r"\pi", met.piNP),
    (r"\epsilon", met.epsilonNP),
    (r"\gamma", met.gammaNP),
], skip_zero=False)

write_named_exprs(fout, "Ricci Tensor Components", [
    (r"\Phi_{00}", met.Phi00NP),
    (r"\Phi_{01}", met.Phi01NP),
    (r"\Phi_{02}", met.Phi02NP),
    (r"\Phi_{10}", met.Phi10NP),
    (r"\Phi_{11}", met.Phi11NP),
    (r"\Phi_{12}", met.Phi12NP),
    (r"\Phi_{20}", met.Phi20NP),
    (r"\Phi_{21}", met.Phi21NP),
    (r"\Phi_{22}", met.Phi22NP),
    (r"\Lambda", met.LambdaNP),
], skip_zero=False)

write_named_exprs(fout, "Weyl Tensor Components", [
    (r"\Psi_0", met.Psi0NP),
    (r"\Psi_1", met.Psi1NP),
    (r"\Psi_2", met.Psi2NP),
    (r"\Psi_3", met.Psi3NP),
    (r"\Psi_4", met.Psi4NP),
], skip_zero=False)

write_equation_section_with_summary(fout, "Newman-Penrose Equations", [
    (r"NP1", met.NPeq1),
    (r"NP2", met.NPeq2),
    (r"NP3", met.NPeq3),
    (r"NP4", met.NPeq4),
    (r"NP5", met.NPeq5),
    (r"NP6", met.NPeq6),
    (r"NP7", met.NPeq7),
    (r"NP8", met.NPeq8),
    (r"NP9", met.NPeq9),
    (r"NP10", met.NPeq10),
    (r"NP11", met.NPeq11),
    (r"NP12", met.NPeq12),
    (r"NP13", met.NPeq13),
    (r"NP14", met.NPeq14),
    (r"NP15", met.NPeq15),
    (r"NP16", met.NPeq16),
    (r"NP17", met.NPeq17),
    (r"NP18", met.NPeq18),
])

write_equation_section_with_summary(fout, "Bianchi Identities", [
    (r"BI1", met.BI1),
    (r"BI2", met.BI2),
    (r"BI3", met.BI3),
    (r"BI4", met.BI4),
    (r"BI5", met.BI5),
    (r"BI6", met.BI6),
    (r"BI7", met.BI7),
    (r"BI8", met.BI8),
    (r"BI9", met.BI9),
    (r"BI10", met.BI10),
    (r"BI11", met.BI11),
])

write_section(fout, "Petrov Type")
petrov = getattr(met, 'petrovtype', None)
if not petrov or petrov.strip() == '':
    try:
        import io, sys,re
        _old_stdout = sys.stdout
        sys.stdout = _buf = io.StringIO()
        met.Petrov_fromWeyl()
        _out = _buf.getvalue()
        sys.stdout = _old_stdout
        _m = re.search(r'Petrov\s*[Tt]ype\s*[=:]?\s*(O|I|II|III|D|N)\b', _out)
        if _m:
            petrov = _m.group(1)
        else:
            petrov = getattr(met, 'petrovtype', 'Unknown')
    except Exception:
        petrov = 'Unknown'
if not petrov or petrov.strip() == '':
    petrov = 'Unknown'
write_display_line(fout, r'\mathrm{Type}\ "' + str(petrov) + r'"')


fout.write(r"\end{document}")
fout.close()

print("tex dosyasi olusturuldu.")

tex dosyasi olusturuldu.
